# TRIAGE-EG E2E-1 — Canonical TEAM-EVAL baseline

Runs frozen Stage2A + T3 + M1 + minimal non-VLM QA. Prediction generation is physically isolated from GT; DEV_CROSS_60 always runs before optional DEV_L21_150.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from time import monotonic
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_EVAL_INPUT = Path(
    os.environ.get(
        "AIC_TEAM_EVAL_DEV_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-team-eval-dev-v1"
    )
)
STAGE1_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
STAGE1E_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1E_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    )
)
CLIP_INPUT = Path(
    os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
)
OPUS_INPUT = Path(
    os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
)
RUN_DEV_CROSS_60 = os.environ.get("AIC_RUN_DEV_CROSS_60", "1") == "1"
RUN_DEV_L21_150 = os.environ.get("AIC_RUN_DEV_L21_150", "1") == "1"
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_e2e1_v01")
RUNTIME_ROOT = Path("/kaggle/working/triage_eg_e2e1_stage2_runtime")
ZIP_PATH = Path("/kaggle/working/triage_eg_e2e1_v01_bundle.zip")
TEAM_EVAL_REPACKED_ZIP = Path("/kaggle/working/aic2026_team_eval_dev_v1_repacked.zip")
EXTRACT_ROOT = Path("/kaggle/working/aic2026_team_eval_dev_v1_extracted")
INFERENCE_ROOT = Path("/kaggle/working/triage_eg_e2e1_inference_only")
STAGE1_MATERIALIZED = Path("/kaggle/working/triage_eg_stage1_materialized")
STAGE1B_MATERIALIZED = Path("/kaggle/working/triage_eg_stage1b_materialized")
STAGE1E_MATERIALIZED = Path("/kaggle/working/triage_eg_stage1e_materialized")
CLIP_MATERIALIZED = Path("/kaggle/working/aic2026_openai_clip_materialized")
OPUS_MATERIALIZED = Path("/kaggle/working/aic2026_opus_materialized")
if not RUN_DEV_CROSS_60:
    raise RuntimeError("DEV_CROSS_60 is mandatory and must execute first")
for path in (
    OUTPUT_ROOT,
    RUNTIME_ROOT,
    EXTRACT_ROOT,
    INFERENCE_ROOT,
    STAGE1_MATERIALIZED,
    STAGE1B_MATERIALIZED,
    STAGE1E_MATERIALIZED,
    CLIP_MATERIALIZED,
    OPUS_MATERIALIZED,
):
    if path.exists():
        if path.parent != Path("/kaggle/working"):
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {path}")
        shutil.rmtree(path)
ZIP_PATH.unlink(missing_ok=True)
TEAM_EVAL_REPACKED_ZIP.unlink(missing_ok=True)
print(
    {
        "required_inputs": {
            "raw_dataset": str(DATA_INPUT),
            "team_eval_dev_bundle": str(TEAM_EVAL_INPUT),
            "stage1_exact_index": str(STAGE1_INPUT),
            "stage1b_verified_contract": str(STAGE1B_INPUT),
            "stage1e_language_contract": str(STAGE1E_INPUT),
            "openai_clip_offline_asset": str(CLIP_INPUT),
            "opus_mt_vi_en_offline_asset": str(OPUS_INPUT),
        },
        "internet_required": "ONLY_FOR_GIT_CLONE_IF_REPO_NOT_PRESENT",
        "model_download_required": False,
        "gpu_policy": "AUTO_CLIP_AND_TRANSLATOR; STAGE1_CPU; OPENCV_CPU; NVDEC_FALSE",
        "run_cross": RUN_DEV_CROSS_60,
        "run_l21": RUN_DEV_L21_150,
        "output_zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")
MAX_DEPTH, MAX_DIRECTORIES = 7, 10000
TEAM_EVAL_REQUIRED_MEMBERS = (
    "README.md",
    "benchmark_registry.json",
    "benchmarks/dev_cross_60/annotation_audit.jsonl",
    "benchmarks/dev_cross_60/gt.jsonl",
    "benchmarks/dev_cross_60/manifest.json",
    "benchmarks/dev_cross_60/queries.jsonl",
    "benchmarks/dev_l21_150/annotation_audit.jsonl",
    "benchmarks/dev_l21_150/gt.jsonl",
    "benchmarks/dev_l21_150/manifest.json",
    "benchmarks/dev_l21_150/queries.jsonl",
)


def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError("Kaggle input discovery exceeded bound")
        yield current
        if depth < MAX_DEPTH:
            queue.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir() and not child.is_symlink()
            )


def resolve_root(hint, marker):
    hint = Path(hint)
    if hint.is_dir() and (hint / marker).is_file():
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory for directory in bounded_dirs(root) if (directory / marker).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one root with {marker}; found {matches}")
    return matches[0]


def resolve_file(hint, filename, *, optional=False):
    hint = Path(hint)
    if hint.is_file() and hint.name == filename:
        return hint.resolve()
    if hint.is_dir() and (hint / filename).is_file():
        return (hint / filename).resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory / filename
            for directory in bounded_dirs(root)
            if (directory / filename).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {filename}; found {matches}")
    return matches[0]


def is_team_eval_root(path):
    path = Path(path)
    return path.is_dir() and all((path / member).is_file() for member in TEAM_EVAL_REQUIRED_MEMBERS)


def resolve_team_eval_root(hint, *, optional=False):
    hint = Path(hint)
    if is_team_eval_root(hint):
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory for directory in bounded_dirs(root) if is_team_eval_root(directory)
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one extracted TEAM-EVAL root; found {matches}")
    return matches[0]


def resolve_dataset(hint):
    hint = Path(hint)
    marker = Path("map-keyframes-aic25-b1/map-keyframes")
    if (hint / marker).is_dir() and any(hint.glob("Videos_*/video")):
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory
            for directory in bounded_dirs(root)
            if (directory / marker).is_dir() and any(directory.glob("Videos_*/video"))
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one raw dataset root; found {matches}")
    return matches[0]


DATASET_ROOT = resolve_dataset(DATA_INPUT)
TEAM_EVAL_MOUNTED_ZIP = resolve_file(TEAM_EVAL_INPUT, "aic2026_team_eval_dev_v1.zip", optional=True)
TEAM_EVAL_MOUNTED_ROOT = resolve_team_eval_root(TEAM_EVAL_INPUT, optional=True)
print(
    {
        "resolved_raw_dataset": str(DATASET_ROOT),
        "team_eval_zip_before_repo_clone": (
            str(TEAM_EVAL_MOUNTED_ZIP) if TEAM_EVAL_MOUNTED_ZIP else "NOT_MOUNTED"
        ),
        "team_eval_extracted_root": (
            str(TEAM_EVAL_MOUNTED_ROOT) if TEAM_EVAL_MOUNTED_ROOT else "NOT_MOUNTED"
        ),
    }
)

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Incomplete repository directory: {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
if not (REPO_DIR / "src/triage_eg/e2e1/pipeline.py").is_file():
    raise RuntimeError(
        "TRIAGEEG ref does not contain E2E-1 implementation; "
        "push repository changes before Kaggle execution"
    )
sys.path.insert(0, str(REPO_DIR / "src"))
HEAD = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
GIT_STATUS = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print({"branch": BRANCH, "HEAD": HEAD, "git_status": GIT_STATUS or "CLEAN"})

In [ ]:
import yaml

from aic2026_eval.io import sha256_file
from triage_eg.e2e1 import E2E1Settings
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root

TEAM_EVAL_FILENAME = "aic2026_team_eval_dev_v1.zip"
REPO_TEAM_EVAL_ZIP = REPO_DIR / "zip" / TEAM_EVAL_FILENAME


def repack_team_eval_root(source_root, destination):
    source_root, destination = Path(source_root), Path(destination)
    missing = [
        member for member in TEAM_EVAL_REQUIRED_MEMBERS if not (source_root / member).is_file()
    ]
    if missing:
        raise RuntimeError(f"Extracted TEAM-EVAL bundle is incomplete; missing: {missing}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.unlink(missing_ok=True)
    with ZipFile(destination, "w", compression=ZIP_DEFLATED) as archive:
        for member in TEAM_EVAL_REQUIRED_MEMBERS:
            if "sealed" in member.casefold():
                raise RuntimeError("SEALED_CONTENT_REJECTED")
            info = ZipInfo(member, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, (source_root / member).read_bytes())
    return destination.resolve(strict=True)


if TEAM_EVAL_MOUNTED_ZIP is not None:
    TEAM_EVAL_ZIP = TEAM_EVAL_MOUNTED_ZIP
    TEAM_EVAL_SOURCE = "KAGGLE_INPUT"
elif TEAM_EVAL_MOUNTED_ROOT is not None:
    TEAM_EVAL_ZIP = repack_team_eval_root(TEAM_EVAL_MOUNTED_ROOT, TEAM_EVAL_REPACKED_ZIP)
    TEAM_EVAL_SOURCE = "KAGGLE_EXTRACTED_BUNDLE_REPACKED"
elif REPO_TEAM_EVAL_ZIP.is_file():
    TEAM_EVAL_ZIP = REPO_TEAM_EVAL_ZIP.resolve()
    TEAM_EVAL_SOURCE = "CLONED_REPOSITORY_FALLBACK"
else:
    raise RuntimeError(
        f"Missing finalized TEAM-EVAL development bundle {TEAM_EVAL_FILENAME}. "
        "Attach a Kaggle dataset containing that exact file or its extracted bundle tree "
        "(nested roots are supported), "
        "or set AIC_TEAM_EVAL_DEV_ROOT to its mounted file/directory. "
        f"Requested mount: {TEAM_EVAL_INPUT}; repository fallback checked: "
        f"{REPO_TEAM_EVAL_ZIP}"
    )
print(
    {
        "resolved_team_eval_zip": str(TEAM_EVAL_ZIP),
        "team_eval_source": TEAM_EVAL_SOURCE,
    }
)


def optional_search_root(hint):
    return None if Path(hint).exists() else SEARCH_ROOT


STAGE1_ROOT = resolve_stage1_root(
    STAGE1_INPUT,
    search_root=optional_search_root(STAGE1_INPUT),
    materialize_root=STAGE1_MATERIALIZED,
)
STAGE1B_ROOT, _ = resolve_input_root(
    STAGE1B_INPUT,
    required=(
        "stage1b_summary.json",
        "encoder/selected_encoder_contract.json",
        "encoder/runtime_adapter_manifest.json",
    ),
    materialize_root=STAGE1B_MATERIALIZED,
    search_root=optional_search_root(STAGE1B_INPUT),
    archive_keyword="stage1b",
)
STAGE1E_ROOT, _ = resolve_input_root(
    STAGE1E_INPUT,
    required=("stage1e_summary.json", "language_path_contract.json"),
    materialize_root=STAGE1E_MATERIALIZED,
    search_root=optional_search_root(STAGE1E_INPUT),
    archive_keyword="stage1e",
)
CLIP_ROOT, _ = resolve_input_root(
    CLIP_INPUT,
    required=("checkpoint/ViT-B-32.pt", "manifests/asset_manifest.json"),
    materialize_root=CLIP_MATERIALIZED,
    search_root=optional_search_root(CLIP_INPUT),
    archive_keyword="clip",
)
OPUS_ROOT, _ = resolve_input_root(
    OPUS_INPUT,
    required=("model/config.json", "manifests/asset_manifest.json"),
    materialize_root=OPUS_MATERIALIZED,
    search_root=optional_search_root(OPUS_INPUT),
    archive_keyword="opus",
)
print(
    {
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
    }
)

GPU_POLICY = yaml.safe_load(
    (REPO_DIR / "configs/retrieval/gpu_g11_frozen_policy.yaml").read_text(encoding="utf-8")
)
E2E_SETTINGS = E2E1Settings()
print("GPU POLICY:", json.dumps(GPU_POLICY, indent=2))
print("E2E FROZEN CONFIG:", json.dumps(E2E_SETTINGS.as_dict(), indent=2))
print("TEAM-EVAL bundle SHA256:", sha256_file(TEAM_EVAL_ZIP))

In [ ]:
from triage_eg.e2e1 import extract_development_bundle

TEAM_EVAL_ROOT = extract_development_bundle(TEAM_EVAL_ZIP, EXTRACT_ROOT)
with ZipFile(TEAM_EVAL_ZIP) as archive:
    MEMBERS = archive.namelist()
assert not any("sealed" in name.casefold() for name in MEMBERS)
CROSS_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_cross_60"
L21_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_l21_150"
print(
    {
        "development_root": str(TEAM_EVAL_ROOT),
        "cross_root": str(CROSS_ROOT),
        "l21_root": str(L21_ROOT),
        "SEALED_ACCESS_GATE": "PASS",
    }
)

In [ ]:
from triage_eg.e2e1 import CanonicalTriagePipeline
from triage_eg.retrieval.stage2 import config_from_yaml

STARTUP_STARTED = monotonic()
STAGE2_CONFIG = config_from_yaml(
    REPO_DIR / "configs/retrieval/stage2_operational_runtime_gpu.yaml",
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1e_root=STAGE1E_ROOT,
    clip_asset_root=CLIP_ROOT,
    translator_asset_root=OPUS_ROOT,
    output_root=RUNTIME_ROOT,
    stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
    build_git_commit=HEAD,
)
PIPELINE = CanonicalTriagePipeline.load_once(STAGE2_CONFIG, DATASET_ROOT, settings=E2E_SETTINGS)
STARTUP_SECONDS = monotonic() - STARTUP_STARTED
print(
    {
        "startup_seconds": STARTUP_SECONDS,
        "resources": PIPELINE.runtime.runtime_manifest(),
        "M3": "DISABLED",
        "EVENT_GRAPH": "DISABLED",
        "VLM": "DISABLED",
        "AGENT": "DISABLED",
        "NVDEC_DEFAULT": False,
    }
)

In [ ]:
from triage_eg.e2e1 import materialize_inference_only, run_prediction_variant

CROSS_INFERENCE = materialize_inference_only(CROSS_ROOT, INFERENCE_ROOT / "dev_cross_60")
assert {path.name for path in CROSS_INFERENCE.iterdir()} == {"queries.jsonl"}
CROSS_P0 = run_prediction_variant(
    PIPELINE, CROSS_INFERENCE, "DEV_CROSS_60", "P0_COARSE", OUTPUT_ROOT
)
print(
    {
        "variant": CROSS_P0["variant"],
        "predictions": len(CROSS_P0["predictions"]),
        "sha256": CROSS_P0["sha256"],
        "validation": CROSS_P0["validation"],
    }
)

In [ ]:
from triage_eg.e2e1 import combine_prediction_variants

CROSS_P1 = run_prediction_variant(
    PIPELINE, CROSS_INFERENCE, "DEV_CROSS_60", "P1_CANONICAL", OUTPUT_ROOT
)
CROSS_RUN = combine_prediction_variants(CROSS_P0, CROSS_P1)
print(
    {
        "variant": CROSS_P1["variant"],
        "predictions": len(CROSS_P1["predictions"]),
        "sha256": CROSS_P1["sha256"],
        "validation": CROSS_P1["validation"],
    }
)

In [ ]:
assert all(value["validation"]["status"] == "PASS" for value in CROSS_RUN["variants"].values())
assert all(
    value["sha256"] == sha256_file(value["prediction_path"])
    for value in CROSS_RUN["variants"].values()
)
print("PREDICTION_CONTRACT_GATE=PASS")
print("GT_LEAKAGE_GATE=PASS")

In [ ]:
from triage_eg.e2e1 import evaluate_finalized

# GT is first loaded inside evaluate_finalized, after both prediction files are hashed.
CROSS_EVAL = evaluate_finalized(CROSS_RUN, CROSS_ROOT, "DEV_CROSS_60", OUTPUT_ROOT)
print(json.dumps({variant: value["summary"] for variant, value in CROSS_EVAL.items()}, indent=2))

In [ ]:
from aic2026_eval.io import write_json
from triage_eg.e2e1 import compare_variants, failure_taxonomy

EVALUATIONS = {"DEV_CROSS_60": CROSS_EVAL}
PREDICTION_RUNS = {"DEV_CROSS_60": CROSS_RUN}
COMPARISON = compare_variants(EVALUATIONS)
TAXONOMY = failure_taxonomy(EVALUATIONS, COMPARISON)
write_json(OUTPUT_ROOT / "diagnostics/p0_vs_p1.json", COMPARISON)
write_json(OUTPUT_ROOT / "diagnostics/failure_taxonomy.json", TAXONOMY)
assert (OUTPUT_ROOT / "evaluation/dev_cross_60_p1_summary.json").is_file()
print("DEV_CROSS_60 persisted before optional L21")

In [ ]:
if RUN_DEV_L21_150:
    L21_INFERENCE = materialize_inference_only(L21_ROOT, INFERENCE_ROOT / "dev_l21_150")
    L21_P0 = run_prediction_variant(
        PIPELINE, L21_INFERENCE, "DEV_L21_150", "P0_COARSE", OUTPUT_ROOT
    )
    L21_P1 = run_prediction_variant(
        PIPELINE, L21_INFERENCE, "DEV_L21_150", "P1_CANONICAL", OUTPUT_ROOT
    )
    L21_RUN = combine_prediction_variants(L21_P0, L21_P1)
    assert all(value["validation"]["status"] == "PASS" for value in L21_RUN["variants"].values())
    PREDICTION_RUNS["DEV_L21_150"] = L21_RUN
    print({"L21_P0_SHA256": L21_P0["sha256"], "L21_P1_SHA256": L21_P1["sha256"]})
else:
    print("DEV_L21_150 prediction run skipped by explicit switch")

In [ ]:
if RUN_DEV_L21_150:
    # L21 GT is first loaded here, after P0/P1 hashes and strict validation.
    L21_EVAL = evaluate_finalized(L21_RUN, L21_ROOT, "DEV_L21_150", OUTPUT_ROOT)
    EVALUATIONS["DEV_L21_150"] = L21_EVAL
    print(json.dumps({variant: value["summary"] for variant, value in L21_EVAL.items()}, indent=2))
COMPARISON = compare_variants(EVALUATIONS)
TAXONOMY = failure_taxonomy(EVALUATIONS, COMPARISON)
write_json(OUTPUT_ROOT / "diagnostics/p0_vs_p1.json", COMPARISON)
write_json(OUTPUT_ROOT / "diagnostics/failure_taxonomy.json", TAXONOMY)
print("TEAM_EVAL_SCORES_REPORTED_SEPARATELY=YES")

In [ ]:
from IPython.display import Image, display

from triage_eg.e2e1 import render_cross_review

REVIEW_PATHS = render_cross_review(PIPELINE, CROSS_RUN, CROSS_EVAL, OUTPUT_ROOT)
assert len([path for path in REVIEW_PATHS if path.parent.name == "review"]) <= 18
for path in [value for value in REVIEW_PATHS if value.parent.name == "montages"]:
    display(Image(filename=str(path)))
print({"review_artifacts": len(REVIEW_PATHS), "GT_OVERLAY_PHASE": "POST_INFERENCE_ONLY"})

In [ ]:
from triage_eg.e2e1 import create_e2e1_bundle, runtime_summary, write_manifests

RUNTIME_SUMMARY = runtime_summary(PREDICTION_RUNS, PIPELINE, STARTUP_SECONDS)
write_manifests(
    OUTPUT_ROOT,
    pipeline=PIPELINE,
    raw_dataset_root=DATASET_ROOT,
    team_eval_bundle=TEAM_EVAL_ZIP,
    branch=BRANCH,
    git_commit=HEAD,
    prediction_runs=PREDICTION_RUNS,
    runtime=RUNTIME_SUMMARY,
)
BUNDLE = create_e2e1_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(BUNDLE) as archive:
    BUNDLE_MEMBERS = archive.namelist()
assert not any(
    name.casefold().endswith((".mp4", ".npy", ".npz", ".pt", ".pth", ".bin"))
    for name in BUNDLE_MEMBERS
)
assert not any(
    any(token in name.casefold() for token in ("sealed", "notebook20", "m3_bundle"))
    for name in BUNDLE_MEMBERS
)
print(
    {
        "download_zip": str(BUNDLE),
        "size_bytes": BUNDLE.stat().st_size,
        "members": len(BUNDLE_MEMBERS),
    }
)

In [ ]:
from triage_eg.e2e1 import formal_report_lines

for line in formal_report_lines(
    git_commit=HEAD,
    evaluations=EVALUATIONS,
    comparison=COMPARISON,
    runtime=RUNTIME_SUMMARY,
    ocr_status=PIPELINE.ocr.status,
    zip_path=BUNDLE,
):
    print(line)
print(
    "INPUTS_USED=",
    {
        "raw_dataset": str(DATASET_ROOT),
        "team_eval_dev_zip": str(TEAM_EVAL_ZIP),
        "team_eval_source": TEAM_EVAL_SOURCE,
        "team_eval_mounted_root": (str(TEAM_EVAL_MOUNTED_ROOT) if TEAM_EVAL_MOUNTED_ROOT else None),
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
    },
)
PIPELINE.close()